In [ ]:
# === Bronze-lager: skapa råtabeller ===
# Tabellerna tar emot data precis som den kommer från API:erna.
import sqlalchemy as db
from sqlalchemy import text
import conn

# Anslutningssträngen hämtas från den lokala modulen conn.py
engine = db.create_engine(conn.get_connection())

queries = [
    # Dagligt genomsnittligt elpris per elområde
    """
    CREATE TABLE IF NOT EXISTS elpris_dagligt_bronze (
        datum DATE,
        datum_code varchar(10),
        elomrade VARCHAR(10),
        genomsnitt_sek_kwh FLOAT
    )
    """,

    # Antal fordon i trafik per region, fordonsslag och år (SCB TAB1278)
    """
    CREATE TABLE IF NOT EXISTS fordon_i_trafik_bronze (
        region_code integer, 
        region_name varchar(50), 
        vehicle_type_code varchar(15), 
        vehicle_type_name varchar(50), 
        year smallint, 
        contents_code varchar(15), 
        value bigint
    )
    """,

    # Nyregistrerade personbilar per region, drivmedel och månad (SCB TAB3277)
    """
    CREATE TABLE IF NOT EXISTS nyregistrerade_personbilar_bronze (
        region_code varchar(15),
        region_name varchar(25),
        fuel_code varchar(20),
        fuel_name varchar(50),
        year smallint,
        month smallint,
        period varchar(10),
        contents_code varchar(15),
        value bigint
    )
    """
] 

# engine.begin() öppnar en transaktion som committas automatiskt om allt lyckas
with engine.begin() as connection:
    for query in queries:
        connection.execute(text(query))

In [ ]:
# === Hämta dagliga elpriser från elprisetjustnu.se ===
import requests
import time
from datetime import date, timedelta

import sqlalchemy as db
from sqlalchemy import text

import conn

# Period och elområde som ska hämtas
START_DATE = date(2022, 1, 1)
END_DATE = date(2025, 12, 31)
PRICE_AREA = "SE3"

current_date = START_DATE
rows = []  # Samlar en rad per dag innan allt skrivs till databasen

# Loopa dag för dag – API:et har en fil per datum och elområde
while current_date <= END_DATE:
    year = current_date.strftime("%Y")
    month_day = current_date.strftime("%m-%d")

    url = (
        f"https://www.elprisetjustnu.se/api/v1/prices/"
        f"{year}/{month_day}_{PRICE_AREA}.json"
    )

    try:
        response = requests.get(url, timeout=30)

        if response.status_code == 200:
            data = response.json()

            # Plocka ut alla kvarts-/timpriser för dagen
            prices = [
                float(row["SEK_per_kWh"])
                for row in data
                if row.get("SEK_per_kWh") is not None
            ]

            # Räkna ut dagens medelpris och spara raden
            if prices:
                daily_average = sum(prices) / len(prices)
                rows.append({
                    "datum": current_date.isoformat(),
                    "datum_code": current_date.isoformat(),
                    "elomrade": PRICE_AREA,
                    "genomsnitt_sek_kwh": daily_average
                })

        # 404 = ingen data publicerad för det datumet
        elif response.status_code == 404:
            print(f"{current_date}: Ingen data")

        else:
            print(f"{current_date}: HTTP {response.status_code}")

    except requests.exceptions.RequestException as e:
        print(f"{current_date}: FEL - {e}")

    current_date += timedelta(days=1)
    time.sleep(0.1)  # Kort paus för att inte överbelasta API:et


engine = db.create_engine(conn.get_connection())


# Skicka in alla rader i ett anrop (executemany) till bronze-tabellen
with engine.begin() as connection:

    connection.execute(
        text("""
            INSERT INTO elpris_dagligt_bronze
            (datum, datum_code, elomrade, genomsnitt_sek_kwh)
            VALUES (
                :datum,
                :datum_code,
                :elomrade,
                :genomsnitt_sek_kwh
            )
        """),
        rows
    )

In [ ]:
# === Hämta "Fordon i trafik" från SCB (tabell TAB1278) ===
import sqlalchemy as db
from sqlalchemy import text

import requests
from itertools import product

import conn

# Metadata beskriver tabellens dimensioner, data-URL:en returnerar själva värdena
metadata_url = (
    "https://statistikdatabasen.scb.se/api/v2/tables/"
    "TAB1278/metadata"
)

data_url = (
    "https://statistikdatabasen.scb.se/api/v2/tables/"
    "TAB1278/data"
)


# 1. Hämta metadata

response = requests.get(
    metadata_url,
    timeout=100
)

response.raise_for_status()

metadata = response.json()

# 2. Läs dimensionerna från metadata (Region, Fordonsslag, ContentsCode, Tid)

region_category = (
    metadata["dimension"]["Region"]["category"]
)

vehicle_category = (
    metadata["dimension"]["Fordonsslag"]["category"]
)

contents_category = (
    metadata["dimension"]["ContentsCode"]["category"]
)

time_category = (
    metadata["dimension"]["Tid"]["category"]
)

# 3. Hämta alla koder per dimension – vi vill ha samtliga värden

region_codes = list(
    region_category["index"].keys()
)

vehicle_codes = list(
    vehicle_category["index"].keys()
)

contents_codes = list(
    contents_category["index"].keys()
)

time_codes = list(
    time_category["index"].keys()
)

# 4. Skapa request
# API:et kräver POST med en JSON-payload för att välja värden (GET med query-parametrar fungerar inte här)


payload = {
    "Selection": [
        {
            "VariableCode": "Region",
            "ValueCodes": region_codes
        },
        {
            "VariableCode": "Fordonsslag",
            "ValueCodes": vehicle_codes
        },
        {
            "VariableCode": "ContentsCode",
            "ValueCodes": contents_codes
        },
        {
            "VariableCode": "Tid",
            "ValueCodes": time_codes
        }
    ],
    "Response": {
        "Format": "json-stat2"
    }
}

# 5. Skicka request

response = requests.post(
    data_url,
    json=payload,
    timeout=100
)

print("HTTP status:", response.status_code)

if response.status_code != 200:
    print(response.text)

response.raise_for_status()

data = response.json()

# 6. Läs dimensionerna i den ordning svaret använder

dimensions = data["id"]

# Sortera koderna efter position så de matchar ordningen i JSON-stat2
dimension_codes = {}

for dimension in dimensions:
    index = data["dimension"][dimension]["category"]["index"]

    # JSON-stat2 anger index som {kod: position}
    codes = [
        code
        for code, position in sorted(
            index.items(),
            key=lambda x: x[1]
        )
    ]

    dimension_codes[dimension] = codes


# Läsbara namn för region och fordonsslag
region_labels = data["dimension"]["Region"]["category"]["label"]
vehicle_labels = data["dimension"]["Fordonsslag"]["category"]["label"]


# 7. JSON-stat2 levererar värdena som en platt lista. Varje värde motsvarar
# en kombination av dimensionerna, så product() återskapar kombinationerna i samma ordning.
combinations = product(
    *[
        dimension_codes[dimension]
        for dimension in dimensions
    ]
)

rows = []

# Para ihop varje kombination med sitt värde och bygg en rad

for combination, value in zip(combinations, data["value"]):

    row = dict(zip(dimensions, combination))

    rows.append({
        "region_code": row["Region"],
        "region_name": region_labels[row["Region"]],
        "vehicle_type_code": row["Fordonsslag"],
        "vehicle_type_name": vehicle_labels[row["Fordonsslag"]],
        "year": int(row["Tid"]),
        "contents_code": row["ContentsCode"],
        "value": int(value) if value is not None else None
    })


# 8. Spara raderna i databasen
engine = db.create_engine(conn.get_connection())


with engine.begin() as connection:

    connection.execute(
        text("""
            INSERT INTO fordon_i_trafik_bronze 
            (region_code, region_name, vehicle_type_code, vehicle_type_name, year, contents_code, value)
            VALUES (
                :region_code,
                :region_name,
                :vehicle_type_code,
                :vehicle_type_name,
                :year,
                :contents_code,
                :value
            )
        """),
        rows
    )

HTTP status: 200


In [ ]:
# === Hämta nyregistrerade personbilar från SCB (tabell TAB3277) ===
import requests
from itertools import product

import sqlalchemy as db 
from sqlalchemy import text

import conn


metadata_url = "https://statistikdatabasen.scb.se/api/v2/tables/TAB3277/metadata"

data_url = "https://statistikdatabasen.scb.se/api/v2/tables/TAB3277/data"


# 1. Hämta metadata

metadata = requests.get(
    metadata_url,
    timeout=30
).json()

# Plocka ut koder och namn för varje dimension
region_codes = list(
    metadata["dimension"]["Region"]["category"]["index"].keys()
)

region_labels = metadata["dimension"]["Region"]["category"]["label"]

fuel_codes = list(
    metadata["dimension"]["Drivmedel"]["category"]["index"].keys()
)

fuel_labels = metadata["dimension"]["Drivmedel"]["category"]["label"]

contents_codes = list(
    metadata["dimension"]["ContentsCode"]["category"]["index"].keys()
)

time_codes = list(
    metadata["dimension"]["Tid"]["category"]["index"].keys()
)

# 2. Hämta data i omgångar (12 perioder åt gången) eftersom API:et begränsar
#    hur många celler ett anrop får returnera

all_rows = []
chunk_size = 12

for start in range(0, len(time_codes), chunk_size):
    chunk_times = time_codes[start:start + chunk_size]
    payload = {
        "Selection": [
            {
                "VariableCode": "Region",
                "ValueCodes": region_codes
            },
            {
                "VariableCode": "Drivmedel",
                "ValueCodes": fuel_codes
            },
            {
                "VariableCode": "ContentsCode",
                "ValueCodes": contents_codes
            },
            {
                "VariableCode": "Tid",
                "ValueCodes": chunk_times
            }
        ],
        "Response": {
            "Format": "json-stat2"
        }
    }
    response = requests.post(
        data_url,
        json=payload,
        timeout=300
    )
    if response.status_code != 200:
        print(response.text)
        response.raise_for_status()

    data = response.json()


    # 3. Läs dimensionerna och sortera koderna efter position

    dimensions = data["id"]
    dimension_codes = {}

    for dimension in dimensions:
        index = data["dimension"][dimension]["category"]["index"]
        codes = [
            code
            for code, position in sorted(
                index.items(),
                key=lambda x: x[1]
            )
        ]
        dimension_codes[dimension] = codes

    # 4. Återskapa varje kombination och para ihop med sitt värde

    combinations = product(
        *[
            dimension_codes[dimension]
            for dimension in dimensions
        ]
    )
    for combination, value in zip(
        combinations,
        data["value"]
    ):
        row = dict(
            zip(dimensions, combination)
        )
        period = row["Tid"]  # Format: "2024M01" -> år = 2024, månad = 01

        all_rows.append({
            "region_code": row["Region"],
            "region_name": region_labels[row["Region"]],
            "fuel_code": row["Drivmedel"],
            "fuel_name": fuel_labels[row["Drivmedel"]],
            "year": int(period[:4]),
            "month": int(period[5:7]),
            "period": period,
            "contents_code": row["ContentsCode"],
            "value": value
        })
    

    # OBS: insert ligger inuti loopen och all_rows töms aldrig, så tidigare
    # omgångar skrivs in igen varje varv (dubbletter). Flytta ut detta block
    # efter loopen, eller insert:a bara den aktuella omgångens rader.
    engine = db.create_engine(conn.get_connection())


    with engine.begin() as connection:

        connection.execute(
            text("""
                INSERT INTO nyregistrerade_personbilar_bronze  
                (region_code, region_name, fuel_code, fuel_name, year, month, period, contents_code, value)
                VALUES (
                    :region_code,
                    :region_name,
                    :fuel_code,
                    :fuel_name,
                    :year,
                    :month,
                    :period,
                    :contents_code,
                    :value
                )
            """),
            all_rows
        )

In [ ]:
# === Silver-lager: rensad och typad data ===
# Tabellerna byggs om från bronze varje körning: DROP + CREATE TABLE AS SELECT.
import sqlalchemy as db
from sqlalchemy import text
import conn

engine = db.create_engine(conn.get_connection())

queries = [
    # Ta bort gamla silver-tabeller så de kan byggas om från grunden
    """
        DROP TABLE IF EXISTS elpris_dagligt_silver;
    """,
    """
        DROP TABLE IF EXISTS fordon_i_trafik_silver;
    """,
    """
        DROP TABLE IF EXISTS nyregistrerade_personbilar_silver;
    """,

    # Elpris: lägg till år/månad, normalisera elområde och filtrera bort ogiltiga priser
    """
        CREATE TABLE elpris_dagligt_silver AS
        SELECT DISTINCT
            CAST(datum AS DATE) AS datum,
            EXTRACT(YEAR FROM datum_code::DATE)::INTEGER AS year,
            EXTRACT(MONTH FROM datum_code::DATE)::INTEGER AS month,
            UPPER(TRIM(elomrade)) AS elomrade,
            CAST(genomsnitt_sek_kwh AS DOUBLE PRECISION) AS genomsnitt_sek_kwh
        FROM elpris_dagligt_bronze
        WHERE datum IS NOT NULL
        AND elomrade IS NOT NULL
        AND TRIM(elomrade) != ''
        AND genomsnitt_sek_kwh IS NOT NULL
        AND genomsnitt_sek_kwh >= 0;
    """,

    # Fordon i trafik: trimma text, säkra datatyper och ta bort tomma/negativa värden
    """
        CREATE TABLE fordon_i_trafik_silver AS
        SELECT DISTINCT
            CAST(region_code AS INT) AS region_code,
            TRIM(region_name) AS region_name,
            TRIM(vehicle_type_code) AS vehicle_type_code,
            TRIM(vehicle_type_name) AS vehicle_type_name,
            CAST(year AS SMALLINT) AS year,
            TRIM(contents_code) AS contents_code,
            CAST(value AS BIGINT) AS value
        FROM fordon_i_trafik_bronze
        WHERE region_code IS NOT NULL
        AND region_name IS NOT NULL
        AND vehicle_type_code IS NOT NULL
        AND vehicle_type_name IS NOT NULL
        AND year IS NOT NULL
        AND contents_code IS NOT NULL
        AND value IS NOT NULL
        AND value >= 0;
    """,

    # Nyregistreringar: samma rensning + kontroll att månaden är 1–12
    """
        CREATE TABLE nyregistrerade_personbilar_silver AS
        SELECT DISTINCT
            CAST(region_code AS INT) AS region_code,
            TRIM(region_name) AS region_name,
            TRIM(fuel_code) AS fuel_code,
            TRIM(fuel_name) AS fuel_name,
            CAST(year AS SMALLINT) AS year,
            CAST(month AS SMALLINT) AS month,
            TRIM(period) AS period,
            TRIM(contents_code) AS contents_code,
            CAST(value AS BIGINT) AS value
        FROM nyregistrerade_personbilar_bronze
        WHERE region_code IS NOT NULL
        AND region_name IS NOT NULL
        AND fuel_code IS NOT NULL
        AND fuel_name IS NOT NULL
        AND year IS NOT NULL
        AND month BETWEEN 1 AND 12
        AND period IS NOT NULL
        AND contents_code IS NOT NULL
        AND value IS NOT NULL
        AND value >= 0;
    """
] 

with engine.begin() as connection:
    for query in queries:
        connection.execute(text(query))

In [ ]:
import sqlalchemy as db
from sqlalchemy import text
import conn

engine = db.create_engine(conn.get_connection())

queries = [
    """
        DROP TABLE IF EXISTS fordon_personbilar_arsvis_gold;
    """,

    """
        CREATE TABLE fordon_personbilar_arsvis_gold AS
        SELECT
            f.year,
            f.region_code,
            f.region_name,
            f.vehicle_type_code,
            f.vehicle_type_name,
            f.value AS antal_fordon,
            p.fuel_code,
            p.fuel_name,
            SUM(p.value) AS antal_nyregistrerade_bilar
        FROM fordon_i_trafik_silver as f
        JOIN nyregistrerade_personbilar_silver as p
            ON f.year = p.year
            AND f.region_code = p.region_code
        GROUP BY
            f.year,
            f.region_code,
            f.region_name,
            f.vehicle_type_code,
            f.vehicle_type_name,
            f.value,
            p.fuel_code,
            p.fuel_name;
    """
]

with engine.begin() as connection:
    for query in queries:
        connection.execute(text(query))

In [ ]:
import sqlalchemy as db
from sqlalchemy import text
import conn

engine = db.create_engine(conn.get_connection())

queries = [
    """
        DROP TABLE IF EXISTS elpris_personbilar_manadsvis_gold;
    """,

    """
    CREATE TABLE elpris_personbilar_manadsvis_gold AS
    SELECT
        e.year,
        e.month,
        e.elomrade,
        AVG(e.genomsnitt_sek_kwh) AS genomsnitt_sek_kwh,
        p.region_code,
        p.region_name,
        p.fuel_code,
        p.fuel_name,
        SUM(p.value) AS antal_nyregistrerade_bilar
    FROM elpris_dagligt_silver e
    JOIN nyregistrerade_personbilar_silver p
        ON e.year = p.year
        AND e.month = p.month
    GROUP BY
        e.year,
        e.month,
        e.elomrade,
        p.region_code,
        p.region_name,
        p.fuel_code,
        p.fuel_name;
    """
]

with engine.begin() as connection:
    for query in queries:
        connection.execute(text(query))

In [ ]:
import sqlalchemy as db
from sqlalchemy import text
import conn

engine = db.create_engine(conn.get_connection())

queries = [
    """
        DROP TABLE IF EXISTS fordon_personbilar_elpris_gold;
    """,

    """
    CREATE TABLE fordon_personbilar_elpris_gold AS
    SELECT
        e.year,
        e.genomsnitt_sek_kwh,
        e.region_code,
        e.region_name,
        f.vehicle_type_code,
        f.vehicle_type_name,
        f.antal_fordon,
        e.fuel_code,
        e.fuel_name,
        e.antal_nyregistrerade_bilar
    FROM elpris_personbilar_manadsvis_gold e
    JOIN fordon_personbilar_arsvis_gold f
        ON e.year = f.year
        AND e.region_code = f.region_code;
    """

]

with engine.begin() as connection:
    for query in queries:
        connection.execute(text(query))

In [ ]:
import sqlalchemy as db
from sqlalchemy import text

import conn



engine = db.create_engine(conn.get_connection())
queries = [
    # """
    #     ALTER TABLE elpris_dagligt 
    #     RENAME TO elpris_dagligt_bronze; 
    # """,

    # """
    #     ALTER TABLE fordon_i_trafik 
    #     RENAME TO fordon_i_trafik_bronze; 
    # """,
    
    # """
    #     ALTER TABLE nyregistrerade_personbilar
    #     RENAME TO nyregistrerade_personbilar_bronze; 
    # """

    # """
    #     drop TABLE elpris_dagligt_bronze 
    # """,

    # """
        # drop TABLE fordon_i_trafik_bronze 
    # """,

    # """
        # drop TABLE nyregistrerade_personbilar_bronze     
    # """
]

with engine.begin() as connection:
    for query in queries:
        connection.execute(text(query))